# Split dataset - train / val

Divise les images annotées (export Label Studio YOLO) en train (80 %) et val (20 %),
puis les copie dans `data/processed/` avec la structure attendue par Ultralytics.

In [1]:
import random
import shutil
from pathlib import Path

SEED = 42
VAL_RATIO = 0.2

PROJECT_ROOT = Path.cwd().parent
SRC_IMAGES = PROJECT_ROOT / "data" / "labels" / "images"
SRC_LABELS = PROJECT_ROOT / "data" / "labels" / "labels"
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Collect stems that have both image and label
stems = sorted([p.stem for p in SRC_IMAGES.glob("*.JPG")
                if (SRC_LABELS / (p.stem + ".txt")).exists()])

print(f"Paires image+label trouvées : {len(stems)}")

Paires image+label trouvées : 225


In [2]:
random.seed(SEED)
random.shuffle(stems)

n_val = int(len(stems) * VAL_RATIO)
val_set = set(stems[:n_val])
trn_set = set(stems[n_val:])

print(f"Train : {len(trn_set)}  |  Val : {len(val_set)}")

Train : 180  |  Val : 45


In [3]:
def copy_split(stems_subset, split_name):
    img_dst = PROCESSED / split_name / "images"
    lbl_dst = PROCESSED / split_name / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for stem in stems_subset:
        shutil.copy(SRC_IMAGES / f"{stem}.JPG", img_dst / f"{stem}.JPG")
        shutil.copy(SRC_LABELS / f"{stem}.txt", lbl_dst / f"{stem}.txt")

    print(f"  {split_name}: {len(stems_subset)} fichiers copiés")


# Réinitialise processed/ pour un split propre
if PROCESSED.exists():
    shutil.rmtree(PROCESSED)

copy_split(trn_set, "train")
copy_split(val_set, "val")
print("Terminé.")

  train: 180 fichiers copiés
  val: 45 fichiers copiés
Terminé.
